In [1]:
from datetime import datetime
import pandas as pd

LOG_COLUMNS = ["experiment_name", "description", "params", "val_accuracy", "val_roc_auc", "notes"]
experiment_log = pd.DataFrame(columns=LOG_COLUMNS)

def log_experiment(experiment_name, description, params=None, accuracy=None, roc_auc=None, notes=""):
    """Append one experiment's results to the in-memory experiment_log table.
    accuracy/roc_auc are optional, so you can log EDA-only findings that never
    reached a trained model, not just full training runs."""
    global experiment_log
    new_row = {
        "val_roc_auc": roc_auc,
        "val_accuracy": accuracy,
        "experiment_name": experiment_name,
        "description": description,
        "params": str(params),
        "notes": notes,
    }
    experiment_log = pd.concat([experiment_log, pd.DataFrame([new_row])], ignore_index=True)
    return experiment_log

In [2]:
# Full experiment history, in chronological order

log_experiment(
    experiment_name="initial_buggy_pipeline",
    description="Original pipeline: preprocessing (imputers, OneHotEncoder) fit separately on train and test, "
                 "transformers fit on the whole train_dataset before splitting into train/val, an index bug in "
                 "the OneHotEncoder join, and id/target accidentally swept into numeric imputation",
    params="n_estimators=1999, learning_rate=0.05, max_depth=5",
    accuracy=0.8995400436813862,
    roc_auc=0.9619856236618058,
    notes="Multiple methodology bugs identified: double-fit preprocessing (data leakage risk), pre-split "
          "leakage, OneHotEncoder index=test_dataset bug, id/target included in imputation columns."
)

log_experiment(
    experiment_name="leakage_fix_no_early_stop",
    description="Fixed double-fit (fit on train only, transform elsewhere) and pre-split leakage (split before "
                 "fitting any transformer). No early stopping yet, n_estimators fixed at 1999.",
    params="n_estimators=1999, learning_rate=0.05, max_depth=5",
    accuracy=0.8995400436813862,
    roc_auc=0.9619820284551825,
    notes="Numbers barely moved vs. the buggy version - confirms the leakage was real but mild. Good sanity "
          "check that fixing methodology issues doesn't always mean a big score swing."
)

log_experiment(
    experiment_name="early_stopping_baseline",
    description="Same clean pipeline, added early_stopping_rounds=50 with n_estimators=5000 as a ceiling "
                 "instead of a fixed tree count",
    params={"n_estimators": 5000, "learning_rate": 0.05, "max_depth": 5, "early_stopping_rounds": 50},
    accuracy=0.9004657419326844,
    roc_auc=0.9626670998979451,
    notes="best_iteration=3634, well short of the 5000 ceiling - confirms the earlier fixed n_estimators=1999 "
          "run had cut training off before it truly converged."
)

log_experiment(
    experiment_name="threshold_tuning_check",
    description="Post-hoc precision-recall threshold sweep on the early_stopping_baseline model (no retraining)",
    params="optimal F1 threshold = 0.4719 vs. default 0.5",
    accuracy=0.90,
    roc_auc=None,
    notes="Essentially flat vs. the default 0.5 threshold - accuracy and macro F1 unchanged. Also a reminder "
          "that threshold tuning can never move ROC-AUC, since AUC already sweeps every threshold internally."
)

log_experiment(
    experiment_name="feature_trim_gpu_scale_pos_weight",
    description="Dropped age/gender/stress_level/academic_work_impact (near-zero permutation importance), "
                 "added GPU device, added scale_pos_weight for class imbalance",
    params={"scale_pos_weight": "neg/pos ratio", "device": "cuda", "n_estimators": 5000},
    accuracy=0.8873974861506864,
    roc_auc=0.9626550256362653,
    notes="scale_pos_weight reshuffled precision/recall a lot (class 0 recall up, class 1 recall down) but "
          "barely moved ROC-AUC - reweighting changes the operating point, not the ranking quality. Feature "
          "trim itself looked safe (permutation importances for the remaining 8 features barely changed)."
)

log_experiment(
    experiment_name="randomizedsearchcv_tuned",
    description="RandomizedSearchCV, 20 candidates x 3-fold CV, scoring=roc_auc, no scale_pos_weight, "
                 "feature-trimmed + GPU",
    params={'subsample': 1.0, 'reg_lambda': 10, 'reg_alpha': 0.1, 'min_child_weight': 7, 'max_depth': 4,
            'learning_rate': 0.1, 'colsample_bytree': 0.8},
    accuracy=0.9006827024603323,
    roc_auc=0.9630148993494188,
    notes="Best confirmed model so far (CV score 0.9631799173024645). Only a ~0.0003 ROC-AUC gain over "
          "early_stopping_baseline - sign that hyperparameter tuning has mostly plateaued for this feature set."
)

log_experiment(
    experiment_name="weekend_gap_feature_eda_check",
    description="Tested weekend_screen_time - daily_screen_time_hours as a candidate feature via a KDE plot "
                 "split by class, before training any model with it",
    params="weekend_gap = weekend_screen_time - daily_screen_time_hours",
    accuracy=None,
    roc_auc=None,
    notes="Class distributions were nearly identical - no separation. Rejected without training a model. "
          "The two raw features were highly correlated (0.80), so subtracting them cancelled out their shared "
          "signal. Next idea: try a sum instead, since summing keeps the shared component."
)
log_experiment(
    experiment_name="combination_weekend_daily_feature_eda_check",
    description="engineered feature added, no measurable improvement, likely within search noise",
    params="combination_weekend_daily = weekend_screen_time + daily_screen_time_hours",
    accuracy=None,
    roc_auc=None,
    notes="Class distributions were provided promising result. Rejected without training a model. "
          "in the permutation importance list, it's in the bottom of the list, the model isn't really relying on it."
)
log_experiment(
    experiment_name="Cross validation 5 folds",
    description="cross validation 5 folds",
    params="",
    accuracy=0.902136,
    roc_auc=0.963523,
    notes="got slightly better result with cross-validation"
)
log_experiment(
    experiment_name="multi_daily_screen_notifiction",
    description="multi = daily_screen_hours * notifications_per_day",
    params="",
    accuracy=0.902136,
    roc_auc=0.963523,
    notes="worse performance"
)
log_experiment(
    experiment_name="using builting imputer",
    description="removed my imputer part, using xgboost builtin imputer",
    params="",
    accuracy=0.902739,
    roc_auc=0.964084,
    notes="better performance"
)
log_experiment(
    experiment_name="using builting imputer with removed multi_daily_screen_notifiction",
    description="removed my imputer part, using xgboost builtin imputer and removed multi_daily_screen_notifiction",
    params="",
    accuracy=0.902903,
    roc_auc=0.964172,
    notes="better performance than previous stage"
)
log_experiment(
    experiment_name="ratio_social_daily",
    description="social_media_hours / daily_screen_time_hours",
    params="",
    accuracy=0.902783,
    roc_auc=0.964243,
    notes="in diagram, there's some overlap, but the peaks are separated; very tiny improvement"
)
log_experiment(
    experiment_name="target_encoding_nested_oof",
    description="Nested out-of-fold target + frequency encoding on all 8 numeric features, added on top "
                 "of the no-imputer baseline. Leak-free: inner 5-fold split builds each training row's "
                 "encoding from other rows only; a single map from the full training fold encodes "
                 "validation and test. Smoothing=10, missing values as their own explicit level.",
    params={'subsample': 1.0, 'reg_lambda': 10, 'reg_alpha': 0.1, 'min_child_weight': 7, 'max_depth': 4,
            'learning_rate': 0.1, 'colsample_bytree': 0.8},
    accuracy=0.905156,
    roc_auc=0.966226,
    notes="Largest, cleanest gain of the project: +0.002054 ROC-AUC over the no-imputer baseline "
          "(0.964172 +/- 0.000501) -- won in all 5 folds by a consistent +0.0018 to +0.0022 margin, "
          "roughly 4x the noise floor on the mean, with tight fold-to-fold variance in the gain itself. "
          "CV std also tightened (0.000437 vs 0.000501). best_iteration dropped sharply (885-1232 vs "
          "2500-3700+), consistent with the model converging on a cleaner signal. Idea sourced from a "
          "public notebook on this competition, where target encoding was the single biggest lever, "
          "bigger than all feature engineering, tuning, and ensembling combined -- confirmed here on our "
          "own model and feature set. New clean baseline going forward: 0.966226 +/- 0.000437."
)
log_experiment(
    experiment_name="target_encoding_nested_oof-without-feature_engineering",
    description="runs the same target_encoding_nested_oof after removing feature engineering i did in previous iterations: sum and ratio",
    params="",
    accuracy=0.904608,
    roc_auc=0.965931,
    notes="roc-auc dropped, however accruacy increased, not submitting the outcome"
)
log_experiment(
    experiment_name="binning_threshold_feature_engineering",
    description="Did feature engineering: binning, threshold",
    params="",
    accuracy=0.904938,
    roc_auc=0.966071,
    notes="not a improvement from the previous one"
)
log_experiment(
    experiment_name="catboost",
    description="only catboost",
    params="",
    accuracy=0.907563,
    roc_auc=0.967101,
    notes="huge improvement"
)
log_experiment(
    experiment_name="catboost with all the features",
    description="only catboost on all the features",
    params="",
    accuracy=0.908017,
    roc_auc=0.967251,
    notes="little less than catboost on selected features; it's okay to run this one"
)
log_experiment(
    experiment_name="xgboost with all the features",
    description="xgboost with all the features",
    params="",
    accuracy=0.904756,
    roc_auc=0.965942,
    notes="worse than selected features"
)
log_experiment(
    experiment_name="combo: 21 xgboost | 79 catboost",
    description="combo: 21 xgboost | 79 catboost; catboost and xgboost for all 10",
    params="",
    accuracy=None,
    roc_auc=0.967582,
    notes="best so far"
)
log_experiment(
    experiment_name="combo: 76.5% catboost, rest xgboost",
    description="combo: 76.5% catboost, rest xgboost | catboost and xgboost for all 10 and eval_metric is AUC for both",
    params="",
    accuracy=None,
    roc_auc=0.967617,
    notes="best so far"
)
experiment_log.sort_values("val_roc_auc", ascending=False)

/tmp/ipykernel_17/2299074117.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  experiment_log = pd.concat([experiment_log, pd.DataFrame([new_row])], ignore_index=True)
/tmp/ipykernel_17/2299074117.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  experiment_log = pd.concat([experiment_log, pd.DataFrame([new_row])], ignore_index=True)
/tmp/ipykernel_17/2299074117.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, th

,experiment_name,description,params,val_accuracy,val_roc_auc,notes
20,"combo: 76.5% catboost, rest xgboost","combo: 76.5% catboost, rest xgboost | catboost...",,NaN,0.967617,best so far
19,combo: 21 xgboost | 79 catboost,combo: 21 xgboost | 79 catboost; catboost and ...,,NaN,0.967582,best so far
17,catboost with all the features,only catboost on all the features,,0.908017,0.967251,little less than catboost on selected features...
16,catboost,only catboost,,0.907563,0.967101,huge improvement
13,target_encoding_nested_oof,Nested out-of-fold target + frequency encoding...,"{'subsample': 1.0, 'reg_lambda': 10, 'reg_alph...",0.905156,0.966226,"Largest, cleanest gain of the project: +0.0020..."
15,binning_threshold_feature_engineering,"Did feature engineering: binning, threshold",,0.904938,0.966071,not a improvement from the previous one
18,xgboost with all the features,xgboost with all the features,,0.904756,0.965942,worse than selected features
14,target_encoding_nested_oof-without-feature_eng...,runs the same target_encoding_nested_oof after...,,0.904608,0.965931,"roc-auc dropped, however accruacy increased, n..."
12,ratio_social_daily,social_media_hours / daily_screen_time_hours,,0.902783,0.964243,"in diagram, there's some overlap, but the peak..."
11,using builting imputer with removed multi_dail...,"removed my imputer part, using xgboost builtin...",,0.902903,0.964172,better performance than previous stage
